# Etape 1 : Index inversé sur 5 phrases

0: "Le hachage est rapide"

1: "Le tri par fusion est stable"

2: "Un index inversé est efficace"

3: "Le hachage et l'index inversé"

4: "Tri rapide vs tri stable"


**Indexation :**

{"le" -> [0, 1, 3]}

{"hachage" -> [0, 3]}

...

# Etape 2 : Nettoyage, indexation, AND

In [2]:
import re
from collections import defaultdict

In [8]:
class MoteurRechercheTextuel:
    def __init__(self):
        # L'index inversé : un dictionnaire où la clé est un mot (str)
        # et la valeur est la liste des IDs de documents (list of int)
        self.index = {}
        
        # Un dictionnaire pour conserver le texte original des documents
        self.documents = {}

    def _nettoyer_texte(self, texte):
        """
        Transforme une chaîne de caractères en une liste de mots nettoyés.
        """
        # 1. Passage en minuscules
        texte_nettoye = texte.lower()
        
        # 2. Remplacement des signes de ponctuation par des espaces
        # (indispensable pour gérer les apostrophes comme dans "l'index")
        pour_remplacement = [".", ",", "!", "?", "'", "-", "«", "»", ":", ";"]
        for caractere in pour_remplacement:
            texte_nettoye = texte_nettoye.replace(caractere, " ")
            
        # 3. Découpage par espace (tokenisation)
        mots = texte_nettoye.split()
        return mots
    
    def indexer_corpus(self, corpus):
        """
        Prend en entrée un dictionnaire { doc_id (int): "texte du document" (str) }
        """
        self.documents = corpus
        
        # On parcourt les documents dans l'ordre de leurs IDs
        for doc_id in sorted(corpus.keys()):
            texte = corpus[doc_id]
            mots = self._nettoyer_texte(texte)
            
            # Utiliser set(mots) évite de dupliquer l'ID si un mot 
            # apparaît plusieurs fois dans le MÊME document.
            for mot in set(mots):
                if mot not in self.index:
                    self.index[mot] = []
                
                # Comme on trie les clés du corpus, doc_id est inséré 
                # de manière strictement croissante !
                self.index[mot].append(doc_id)
    
    def _intersecter_deux_listes(self, liste1, liste2):
        """
        Algorithme à deux pointeurs pour intersecter deux listes triées.
        Complexité optimale : O(|liste1| + |liste2|)
        """
        intersection = []
        i, j = 0, 0  # Nos deux pointeurs

        while i < len(liste1) and j < len(liste2):
            if liste1[i] == liste2[j]:
                intersection.append(liste1[i])
                i += 1
                j += 1
            elif liste1[i] < liste2[j]:
                i += 1  # On avance le pointeur de la liste 1 car sa valeur est trop petite
            else:
                j += 1  # On avance le pointeur de la liste 2
                
        return intersection

    def requete_and(self, requete_texte):
        """
        Traite une requête multi-mots comme un AND logique.
        Exemple : "hachage inversé" -> cherche les docs avec les DEUX mots.
        """
        mots_requete = self._nettoyer_texte(requete_texte)
        if not mots_requete:
            return []

        # Si l'un des mots de la requête n'est pas du tout dans l'index,
        # l'intersection AND est forcément vide.
        for mot in mots_requete:
            if mot not in self.index:
                return []

        # On initialise nos résultats avec la liste du premier mot
        resultats = self.index[mots_requete[0]]

        # On intersecte successivement avec les listes des autres mots
        for mot in mots_requete[1:]:
            liste_suivante = self.index[mot]
            resultats = self._intersecter_deux_listes(resultats, liste_suivante)

        return resultats

In [ ]:
if __name__ == "__main__":
    # Notre mini-corpus de test
    mini_corpus = {
        0: "Le hachage est rapide",
        1: "Le tri par fusion est stable",
        2: "Un index inversé est efficace",
        3: "Le hachage et l'index inversé",
        4: "Tri rapide vs tri stable"
    }

    # Instanciation et indexation
    moteur = MoteurRechercheTextuel()
    moteur.indexer_corpus(mini_corpus)

    print("--- 1. Vérification de l'Index Inversé ---")
    print(f"Posting list pour 'hachage' : {moteur.index.get('hachage')}") # Attendu: [0, 3]
    print(f"Posting list pour 'rapide'  : {moteur.index.get('rapide')}")  # Attendu: [0, 4]
    
    print("\n--- 2. Test des requêtes AND ---")
    print(f"Requête 'hachage rapide' : {moteur.requete_and('hachage rapide')}") # Attendu: [0]
    print(f"Requête 'tri stable'     : {moteur.requete_and('tri stable')}")     # Attendu: [1, 4]
    print(f"Requête 'hachage inversé' : {moteur.requete_and('hachage inversé')}") # Attendu: [3]

--- 1. Vérification de l'Index Inversé ---
Posting list pour 'hachage' : [0, 3]
Posting list pour 'rapide'  : [0, 4]

--- 2. Test des requêtes AND ---
Requête 'hachage rapide' : [0]
Requête 'tri stable'     : [1, 4]
Requête 'hachage inversé' : [3]


In [6]:
print(f"Requête 'tri stable rapide' : {moteur.requete_and('tri stable rapide')}")

Requête 'tri stable rapide' : [4]


# Etape 3 : TF-IDF

In [9]:
class MoteurRechercheTextuel_TFIDF:
    def __init__(self):
        # L'index inversé enrichi : { terme -> [(doc_id, frequence), (doc_id, frequence), ...] }
        self.index = {}
        # Stockage des textes originaux : { doc_id -> "texte" }
        self.documents = {}
        # Stockage des longueurs de documents : { doc_id -> nombre_de_mots }
        self.tailles_documents = {}

    def _nettoyer_texte(self, texte):
        """Transforme une chaîne en liste de mots nettoyés."""
        texte_nettoye = texte.lower()
        for caractere in [".", ",", "!", "?", "'", "-", "«", "»", ":", ";"]:
            texte_nettoye = texte_nettoye.replace(caractere, " ")
        return texte_nettoye.split()

    def indexer_corpus(self, corpus):
        """
        Indexe le corpus en calculant les fréquences locales (TF) 
        et la taille de chaque document.
        """
        self.documents = corpus
        
        for doc_id in sorted(corpus.keys()):
            texte = corpus[doc_id]
            mots = self._nettoyer_texte(texte)
            
            # 1. Enregistrement de la longueur du document
            self.tailles_documents[doc_id] = len(mots)
            
            if len(mots) == 0:
                continue
                
            # 2. Comptage de la fréquence de chaque mot dans CE document
            # On utilise un dictionnaire temporaire pour ce document précis
            frequences_locales = {}
            for mot in mots:
                frequences_locales[mot] = frequences_locales.get(mot, 0) + 1
            
            # 3. Insertion dans l'index inversé global
            for mot, freq in frequences_locales.items():
                if mot not in self.index:
                    self.index[mot] = []
                
                # On ajoute le couple (doc_id, frequence)
                # Comme doc_id est trié à la base, nos listes restent triées par ID !
                self.index[mot].append((doc_id, freq))

    def _intersecter_deux_listes(self, liste1, liste2):
        """
        Intersection de deux listes de couples (doc_id, freq) via deux pointeurs.
        On compare uniquement les doc_id (index 0 du couple).
        """
        intersection = []
        i, j = 0, 0

        while i < len(liste1) and j < len(liste2):
            doc_id1 = liste1[i][0] # On récupère l'ID du doc
            doc_id2 = liste2[j][0]
            
            if doc_id1 == doc_id2:
                # En cas de correspondance, on peut choisir de garder 
                # les infos du document (on verra au moment du scoring)
                intersection.append(liste1[i]) 
                i += 1
                j += 1
            elif doc_id1 < doc_id2:
                i += 1
            else:
                j += 1
                
        return intersection

    def requete_and(self, requete_texte):
        """Renvoie les couples (doc_id, freq) des documents contenant tous les mots."""
        mots_requete = self._nettoyer_texte(requete_texte)
        if not mots_requete:
            return []

        for mot in mots_requete:
            if mot not in self.index:
                return []

        resultats = self.index[mots_requete[0]]

        for mot in mots_requete[1:]:
            liste_suivante = self.index[mot]
            resultats = self._intersecter_deux_listes(resultats, liste_suivante)

        return resultats

In [10]:
if __name__ == "__main__":
    mini_corpus = {
        0: "Le hachage est rapide, très rapide", # "rapide" est présent 2 fois
        1: "Le tri par fusion est stable",
        2: "Un index inversé est efficace",
        3: "Le hachage et l'index inversé",
        4: "Tri rapide vs tri stable"
    }

    moteur = MoteurRechercheTextuel_TFIDF()
    moteur.indexer_corpus(mini_corpus)

    print("--- 1. VÉRIFICATION DES TAILLES ---")
    print(f"Nombre de mots dans Doc 0 : {moteur.tailles_documents[0]}") # Attendu : 5
    print(f"Nombre de mots dans Doc 4 : {moteur.tailles_documents[4]}") # Attendu : 5

    print("\n--- 2. VÉRIFICATION DE L'INDEX ENRICHI ---")
    # Attendu : [(0, 1), (3, 1)] -> Présent dans doc 0 (1 fois) et doc 3 (1 fois)
    print(f"Posting list pour 'hachage' : {moteur.index.get('hachage')}") 
    
    # Attendu : [(0, 2), (4, 1)] -> Présent dans doc 0 (2 fois) et doc 4 (1 fois)
    print(f"Posting list pour 'rapide'  : {moteur.index.get('rapide')}") 

    print("\n--- 3. TEST REQUÊTE AND ADAPTÉE ---")
    print(f"Requête 'hachage rapide' : {moteur.requete_and('hachage rapide')}") 
    # Attendu : [(0, 1)] (car seul le doc 0 contient les deux mots)

--- 1. VÉRIFICATION DES TAILLES ---
Nombre de mots dans Doc 0 : 6
Nombre de mots dans Doc 4 : 5

--- 2. VÉRIFICATION DE L'INDEX ENRICHI ---
Posting list pour 'hachage' : [(0, 1), (3, 1)]
Posting list pour 'rapide'  : [(0, 2), (4, 1)]

--- 3. TEST REQUÊTE AND ADAPTÉE ---
Requête 'hachage rapide' : [(0, 1)]


# Etape 4 : Calcul de la pertinence et le classement

Pour une requête composée de plusieurs mots (ex: "hachage rapide"), le score d'un document $d$ est la somme des scores TF-IDF de chaque mot $t$ de la requête présent dans ce document :$$\text{Score}_{\text{TF-IDF}}(q, d) = \sum_{t \in q} \text{TF}(t, d) \times \text{IDF}(t)$$

Puisque nous codons from scratch, voici les formules exactes que nous allons implémenter :

Le TF (Term Frequency) normalisé :$$\text{TF}(t, d) = \frac{\text{Fréquence brute du mot } t \text{ dans } d}{\text{Nombre total de mots dans } d}$$

L'IDF (Inverse Document Frequency) standard :$$\text{IDF}(t) = \log\left(\frac{N}{\text{df}_t}\right)$$

$N$ : Le nombre total de documents dans le corpus (égal à len(self.documents)).

$\text{df}_t$ : Le nombre de documents contenant le mot $t$ (égal à la longueur de la posting list du mot $t$).

Note : Pour éviter les divisions par zéro si un mot n'est pas dans le corpus, on ajoute généralement $+1$, ce qui donne : $\log\left(\frac{N}{\text{df}_t + 1}\right) + 1$. On utilise la fonction math.log (logarithme népérien).

In [1]:
import math

class MoteurRechercheTextuel_score:
    def __init__(self):
        self.index = {}  # { terme -> [(doc_id, freq), ...] }
        self.documents = {}  # { doc_id -> "texte" }
        self.tailles_documents = {}  # { doc_id -> nombre_de_mots }

    def _nettoyer_texte(self, texte):
        texte_nettoye = texte.lower()
        for caractere in [".", ",", "!", "?", "'", "-", "«", "»", ":", ";"]:
            texte_nettoye = texte_nettoye.replace(caractere, " ")
        return texte_nettoye.split()

    def indexer_corpus(self, corpus):
        self.documents = corpus
        for doc_id in sorted(corpus.keys()):
            mots = self._nettoyer_texte(corpus[doc_id])
            self.tailles_documents[doc_id] = len(mots)
            if len(mots) == 0:
                continue
                
            frequences_locales = {}
            for mot in mots:
                frequences_locales[mot] = frequences_locales.get(mot, 0) + 1
            
            for mot, freq in frequences_locales.items():
                if mot not in self.index:
                    self.index[mot] = []
                self.index[mot].append((doc_id, freq))

    def _intersecter_deux_listes(self, liste1, liste2):
        intersection = []
        i, j = 0, 0
        while i < len(liste1) and j < len(liste2):
            if liste1[i][0] == liste2[j][0]:
                intersection.append(liste1[i]) 
                i += 1
                j += 1
            elif liste1[i][0] < liste2[j][0]:
                i += 1
            else:
                j += 1
        return intersection

    def requete_and(self, requete_texte):
        mots_requete = self._nettoyer_texte(requete_texte)
        if not mots_requete:
            return []
        for mot in mots_requete:
            if mot not in self.index:
                return []
        resultats = self.index[mots_requete[0]]
        for mot in mots_requete[1:]:
            resultats = self._intersecter_deux_listes(resultats, self.index[mot])
        return resultats

    # ==================== NOUVEAUTÉS ÉTAPE 4 ====================

    def _calculer_idf(self, terme):
        """Calcule l'IDF d'un terme avec l'ajustement pour éviter les divisions par 0."""
        N = len(self.documents)
        # df_t est la longueur de la posting list pour ce terme
        df_t = len(self.index.get(terme, []))
        
        if df_t == 0:
            return 0.0
            
        # Formule lissée classique
        return math.log((N / (df_t + 1))) + 1

    def chercher_tfidf(self, requete_texte):
        """
        Exécute la requête AND, calcule le score TF-IDF des documents correspondants
        et les renvoie triés par pertinence décroissante.
        """
        mots_requete = self._nettoyer_texte(requete_texte)
        
        # 1. On récupère d'abord les documents candidats (ceux qui contiennent TOUS les mots)
        documents_candidats = self.requete_and(requete_texte)
        if not documents_candidats:
            return []

        # Pour faciliter le calcul, on extrait juste la liste des doc_ids uniques filtrés
        doc_ids_filtrés = [couple[0] for couple in documents_candidats]

        # 2. Calcul du score pour chaque document candidat
        scores_documents = {}
        
        # On calcule à l'avance l'IDF de chaque mot de la requête pour gagner du temps
        idfs_requete = {mot: self._calculer_idf(mot) for mot in mots_requete}

        for doc_id in doc_ids_filtrés:
            score_total = 0.0
            
            for mot in mots_requete:
                # On doit retrouver la fréquence brute du mot dans ce document précis
                # On cherche dans la posting list du mot
                frequence_brute = 0
                for d_id, freq in self.index[mot]:
                    if d_id == doc_id:
                        frequence_brute = freq
                        break
                
                # Calcul du TF normalisé pour ce document
                taille_doc = self.tailles_documents[doc_id]
                tf = frequence_brute / taille_doc
                
                # Ajout au score cumulé du document
                score_total += tf * idfs_requete[mot]
                
            scores_documents[doc_id] = score_total

        # 3. Tri des documents par score décroissant
        # sorted renvoie une liste de couples (doc_id, score)
        resultats_tries = sorted(scores_documents.items(), key=lambda x: x[1], reverse=True)
        
        return resultats_tries

In [2]:
if __name__ == "__main__":
    mini_corpus = {
        0: "Le hachage hachage hachage est rapide, très rapide",  # Très dense en mots-clés
        1: "Le tri par fusion est stable",
        2: "Un index inversé est efficace",
        3: "Le hachage et l'index inversé sont une solution mais le processus n'est pas toujours rapide", # Long et dilué
        4: "Tri rapide vs tri stable"
    }

    moteur = MoteurRechercheTextuel_score()
    moteur.indexer_corpus(mini_corpus)

    print("--- TEST DU RANKING TF-IDF ---")
    requete = "hachage rapide"
    resultats = moteur.chercher_tfidf(requete)
    
    print(f"Résultats pour la recherche '{requete}' (triés par score) :")
    for doc_id, score in resultats:
        print(f" - [Doc {doc_id}] Score: {score:.4f} | Texte: '{moteur.documents[doc_id]}'")

--- TEST DU RANKING TF-IDF ---
Résultats pour la recherche 'hachage rapide' (triés par score) :
 - [Doc 0] Score: 0.8723 | Texte: 'Le hachage hachage hachage est rapide, très rapide'
 - [Doc 3] Score: 0.1608 | Texte: 'Le hachage et l'index inversé sont une solution mais le processus n'est pas toujours rapide'


Le Doc 0 obtient un score nettement supérieur au Doc 3 car dans le Doc 0, les mots recherchés représentent une part immense du document (TF très élevé), alors que dans le Doc 3, les mots sont "noyés" au milieu d'une phrase longue, leur TF est donc écrasé par la taille du document.

Limite observé : Le problème du TF-IDF est : le score grandit de manière linéaire avec la fréquence du mot, donc si un mot apparaît 20 fois dans un document, le score est multiplié par 20. Hors, en réalité, la pertinence s'essouffle : il y a un effet de saturation. L'écart est énorme entre 1 et 2 apparitions d'un mot, mais il est presque invisible entre 10 et 11. En plus, le TF-IDF ne prend pas bien en compte la longueur des textes, ce qui favorise trop les documents longs. C'est là que le BM25 intervient : il règle ces deux problèmes en bloquant la hausse infinie du score (grâce au paramètre $k_1$) et en ajustant le calcul selon la taille moyenne des documents du corpus (avec le paramètre $b$).

# Etape 5 : BM25

Pour un document $d$ et une requête $q$, le score BM25 se calcule ainsi :$$\text{Score}_{\text{BM25}}(q, d) = \sum_{t \in q} \text{IDF}_{\text{BM25}}(t) \times \frac{f_{t,d} \times (k_1 + 1)}{f_{t,d} + k_1 \times \left(1 - b + b \times \frac{L_d}{L_{\text{avg}}}\right)}$$

Où :$f_{t,d}$ : La fréquence brute du mot $t$ dans le document $d$.

$L_d$ : La longueur du document $d$ (en nombre de mots).

$L_{\text{avg}}$ : La longueur moyenne de tous les documents du corpus.

$k_1$ : Le paramètre de saturation du TF (on prend souvent 1.5).$b$ : Le paramètre de pénalisation de la longueur (on prend souvent 0.75).

Et pour l'IDF version BM25, la formule standard est un peu différente du TF-IDF classique pour éviter les scores négatifs sur les mots très fréquents :$$\text{IDF}_{\text{BM25}}(t) = \log\left(\frac{N - \text{df}_t + 0.5}{\text{df}_t + 0.5} + 1\right)$$

In [2]:
import math

class MoteurRechercheTextuel_BM25:
    def __init__(self):
        self.index = {}  # { terme -> [(doc_id, freq), ...] }
        self.documents = {}  # { doc_id -> "texte" }
        self.tailles_documents = {}  # { doc_id -> nombre_de_mots }
        self.longueur_moyenne = 0.0  # L_avg pour le BM25

    def _nettoyer_texte(self, texte):
        texte_nettoye = texte.lower()
        for caractere in [".", ",", "!", "?", "'", "-", "«", "»", ":", ";"]:
            texte_nettoye = texte_nettoye.replace(caractere, " ")
        return texte_nettoye.split()

    def indexer_corpus(self, corpus):
        self.documents = corpus
        total_mots = 0
        
        for doc_id in sorted(corpus.keys()):
            mots = self._nettoyer_texte(corpus[doc_id])
            self.tailles_documents[doc_id] = len(mots)
            total_mots += len(mots)
            
            if len(mots) == 0:
                continue
                
            frequences_locales = {}
            for mot in mots:
                frequences_locales[mot] = frequences_locales.get(mot, 0) + 1
            
            for mot, freq in frequences_locales.items():
                if mot not in self.index:
                    self.index[mot] = []
                self.index[mot].append((doc_id, freq))
        
        # Calcul de la longueur moyenne du corpus (indispensable pour BM25)
        if len(corpus) > 0:
            self.longueur_moyenne = total_mots / len(corpus)

    def _intersecter_deux_listes(self, liste1, liste2):
        intersection = []
        i, j = 0, 0
        while i < len(liste1) and j < len(liste2):
            if liste1[i][0] == liste2[j][0]:
                intersection.append(liste1[i]) 
                i += 1
                j += 1
            elif liste1[i][0] < liste2[j][0]:
                i += 1
            else:
                j += 1
        return intersection

    def requete_and(self, requete_texte):
        mots_requete = self._nettoyer_texte(requete_texte)
        if not mots_requete:
            return []
        for mot in mots_requete:
            if mot not in self.index:
                return []
        resultats = self.index[mots_requete[0]]
        for mot in mots_requete[1:]:
            resultats = self._intersecter_deux_listes(resultats, self.index[mot])
        return resultats

    # ==================== NOUVEAUTÉS ÉTAPE 5 : BM25 ====================

    def _calculer_idf_bm25(self, terme):
        """Calcule l'IDF version BM25 pour éviter les scores négatifs."""
        N = len(self.documents)
        df_t = len(self.index.get(terme, []))
        if df_t == 0:
            return 0.0
        # Formule de l'IDF Okapi BM25
        return math.log(((N - df_t + 0.5) / (df_t + 0.5)) + 1)

    def chercher_bm25(self, requete_texte, k1=1.5, b=0.75):
        """
        Exécute la requête AND, calcule le score BM25 des documents
        et les renvoie triés par pertinence décroissante.
        """
        mots_requete = self._nettoyer_texte(requete_texte)
        
        # 1. Récupération des documents qui passent le filtre AND
        documents_candidats = self.requete_and(requete_texte)
        if not documents_candidats:
            return []

        doc_ids_filtrés = [couple[0] for couple in documents_candidats]
        scores_documents = {}
        
        # Pré-calcul des IDF BM25 pour la requête
        idfs_bm25 = {mot: self._calculer_idf_bm25(mot) for mot in mots_requete}

        # 2. Calcul du score BM25 pour chaque document
        for doc_id in doc_ids_filtrés:
            score_total = 0.0
            L_d = self.tailles_documents[doc_id] # Longueur du doc actuel
            
            for mot in mots_requete:
                # Récupération de la fréquence brute du mot dans ce doc
                frequence_brute = 0
                for d_id, freq in self.index[mot]:
                    if d_id == doc_id:
                        frequence_brute = freq
                        break
                
                # Application de la formule de régulation du TF de BM25
                numerateur = frequence_brute * (k1 + 1)
                denominateur = frequence_brute + k1 * (1 - b + b * (L_d / self.longueur_moyenne))
                
                # Score partiel pour ce mot
                score_mot = idfs_bm25[mot] * (numerateur / denominateur)
                score_total += score_mot
                
            scores_documents[doc_id] = score_total

        # 3. Tri des résultats par score décroissant
        return sorted(scores_documents.items(), key=lambda x: x[1], reverse=True)

In [4]:
if __name__ == "__main__":
    mini_corpus = {
        0: "Le hachage hachage hachage est rapide rapide", 
        1: "Le tri par fusion est stable",
        2: "Un index inversé est efficace",
        3: "Le hachage et l'index inversé sont une solution mais le processus n'est pas toujours rapide", 
        4: "Tri rapide vs tri stable"
    }

    moteur = MoteurRechercheTextuel_BM25()
    moteur.indexer_corpus(mini_corpus)

    print("--- COMPARISON TF-IDF VS BM25 ---")
    requete = "hachage rapide"
    
    print("\n[Résultats BM25]")
    resultats_bm25 = moteur.chercher_bm25(requete)
    for doc_id, score in resultats_bm25:
        print(f" - [Doc {doc_id}] Score: {score:.4f} | '{moteur.documents[doc_id]}'")

--- COMPARISON TF-IDF VS BM25 ---

[Résultats BM25]
 - [Doc 0] Score: 2.3084 | 'Le hachage hachage hachage est rapide rapide'
 - [Doc 3] Score: 0.9391 | 'Le hachage et l'index inversé sont une solution mais le processus n'est pas toujours rapide'


Si on avait utilisé un TF classique non normalisé, le Doc 3 (qui contient les deux mots) aurait pu avoir un score proche du Doc 0 car les mots y sont présents.

Le BM25 change la donne :

Il récompense le Doc 0 parce qu'il est court et que les mots-clés y sont ultra-concentrés.

Il bride le Doc 0 pour que ses répétitions ("hachage hachage...") ne fassent pas s'envoler le score de manière artificielle.

Il pénalise le Doc 3 car les mots-clés ne représentent qu'une infime fraction du texte total (effet de dilution).

# Etape 5 : BM25

BM25 est une autre méthode de ranking. Elle garde l'idée de favoriser les termes rares, mais elle ajuste aussi le score avec une saturation de fréquence et une normalisation par la longueur du document.

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))
from lib.MoteurRechercheTextuel import MoteurRechercheTextuel

mini_corpus = {
    0: "Le hachage hachage hachage est rapide, très rapide",
    1: "Le tri par fusion est stable",
    2: "Un index inversé est efficace",
    3: "Le hachage et l'index inversé sont une solution, mais le processus n'est pas toujours rapide",
    4: "Tri rapide vs tri stable",
    5: ""
}

moteur = MoteurRechercheTextuel()
moteur.indexer_corpus(mini_corpus)

requete = "hachage rapide"

print("--- COMPARAISON TF-IDF / BM25 ---")
print(f"Longueur moyenne des documents : {moteur.longueur_moyenne_documents:.2f} mots")

print("\nTF-IDF :")
for doc_id, score in moteur.chercher_tfidf(requete):
    print(f" - Doc {doc_id} | score={score:.4f} | {moteur.documents[doc_id]}")

print("\nBM25 :")
for doc_id, score in moteur.chercher_bm25(requete):
    print(f" - Doc {doc_id} | score={score:.4f} | {moteur.documents[doc_id]}")

--- COMPARAISON TF-IDF / BM25 ---
Longueur moyenne des documents : 8.20 mots

TF-IDF :
 - Doc 0 | score=0.9863 | Le hachage hachage hachage est rapide, très rapide
 - Doc 3 | score=0.1823 | Le hachage et l'index inversé sont une solution, mais le processus n'est pas toujours rapide

BM25 :
 - Doc 0 | score=2.7246 | Le hachage hachage hachage est rapide, très rapide
 - Doc 3 | score=1.1617 | Le hachage et l'index inversé sont une solution, mais le processus n'est pas toujours rapide
